# Benchmark di Controllo AID: Reiezione di Pasti Non Annunciati

Questo notebook implementa e confronta due architetture di controllo a circuito chiuso su ambiente di simulazione *in silico* basato su `simglucose` (modello metabolico UVA/Padova):
1. **PID Baseline Convenzionale**: regolatore proporzionale-integrale-derivativo non vincolato, accordato sul setpoint euglicemico ($110\text{ mg/dL}$).
2. **PID con Vincolo su IOB e Modulo PLGS**: regolatore a retroazione dotato di saturazione dinamica basata sulla stima dell'insulina attiva residua ($IOB(t)$) e sospensione predittiva dell'infusione (*Predictive Low-Glucose Suspend*).

### Dettagli dello Scenario di Simulazione
- **Paziente virtuale**: `adult#001`
- **Durata**: $16\text{ ore}$ nominali (dalle ore 08:00 alle 00:00)
- **Passo di campionamento ($dt$)**: $3\text{ minuti}$ (`env.sample_time = 3`), per un totale di $320\text{ passi}$
- **Disturbo esogeno**: pasto non annunciato da $60\text{ g}$ di carboidrati alle ore 12:00 ($t = 240\text{ min}$, passo 80)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from simglucose.simulation.env import T1DSimEnv
from simglucose.actuator.pump import InsulinPump
from simglucose.sensor.cgm import CGMSensor
from simglucose.patient.t1dpatient import T1DPatient
from simglucose.simulation.scenario import CustomScenario
from simglucose.controller.base import Controller, Action

print("Ambiente e librerie caricate con successo.")

## 1. Definizione delle Leggi di Controllo

### Correzione Discretizzazione con $dt = 3\text{ min}$
Nel simulatore `simglucose`, il sensore Dexcom campiona ogni $3\text{ minuti}$. È fondamentale moltiplicare i termini integrativi e differenziali per $dt = 3.0$ per preservare il significato fisico e dimensionale delle grandezze.

In [ ]:
class StandardPIDController(Controller):
    def __init__(self, target=110.0, kp=0.0006, ki=0.000015, kd=0.004, basal_rate=0.021, dt=3.0):
        self.target = target
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.basal_rate = basal_rate
        self.dt = dt
        self.integral_error = 0.0
        self.prev_cgm = None

    def policy(self, observation, reward, done, **info):
        cgm = observation.CGM
        error = cgm - self.target
        
        # Azione integrale con anti-windup (integrale in mg/dL * min)
        self.integral_error += error * self.dt
        self.integral_error = float(np.clip(self.integral_error, -3000.0, 10000.0))
        
        # Azione derivativa su CGM misurato (variazione per minuto)
        if self.prev_cgm is None:
            d_cgm_dt = 0.0
        else:
            d_cgm_dt = (cgm - self.prev_cgm) / self.dt
        self.prev_cgm = cgm
        
        # Calcolo comando PID
        correction = (self.kp * error) + (self.ki * self.integral_error) + (self.kd * d_cgm_dt)
        u = self.basal_rate + correction
        u = max(0.0, min(u, 0.25))  # Saturazione fisica attuatore [0, 0.25] U/min
        
        return Action(basal=u, bolus=0.0)

    def reset(self):
        self.integral_error = 0.0
        self.prev_cgm = None


class IOBConstrainedController(Controller):
    def __init__(self, target=110.0, kp=0.0006, ki=0.000015, kd=0.004, basal_rate=0.021, 
                 tau_s=50.0, iob_cap=2.2, plgs_threshold=90.0, dt=3.0):
        self.target = target
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.basal_rate = basal_rate
        self.tau_s = tau_s
        self.iob_cap = iob_cap
        self.plgs_threshold = plgs_threshold
        self.dt = dt
        
        self.integral_error = 0.0
        self.prev_cgm = None
        self.s1 = 0.0
        self.s2 = 0.0
        self.current_iob = 0.0

    def policy(self, observation, reward, done, **info):
        cgm = observation.CGM
        error = cgm - self.target
        
        # 1. Aggiornamento osservatore IOB a due compartimenti (Dalla Man / Hovorka)
        # u_excess rappresenta l'insulina infusa oltre la basale
        self.current_iob = (self.s1 + self.s2) / 60.0  # IOB stimato in Unità [U]
        
        # Derivata CGM
        if self.prev_cgm is None:
            d_cgm_dt = 0.0
        else:
            d_cgm_dt = (cgm - self.prev_cgm) / self.dt
        
        # 2. Modulo PLGS: Proiezione a 20 minuti
        cgm_projected_20min = cgm + (d_cgm_dt * 20.0)
        
        # 3. Azione integrale con sospensione se in discesa o vicino al target
        if cgm > self.target:
            self.integral_error += error * self.dt
        else:
            self.integral_error = max(0.0, self.integral_error - (abs(error) * self.dt * 0.5))
        self.integral_error = float(np.clip(self.integral_error, -1000.0, 6000.0))
        
        # 4. Calcolo correzione PID
        correction = (self.kp * error) + (self.ki * self.integral_error) + (self.kd * d_cgm_dt)
        u = self.basal_rate + correction
        
        # 5. Applicazione del vincolo IOB (saturazione dinamica per evitare insulin stacking)
        if self.current_iob > self.iob_cap and u > self.basal_rate:
            u = self.basal_rate  # Blocca le dosi correttive se l'insulina residua è già elevata
            
        # 6. Intervento PLGS di sicurezza ipoglicemica
        if cgm_projected_20min < self.plgs_threshold or cgm < 75.0:
            u = 0.0  # Sospensione totale dell'erogazione
            
        u = max(0.0, min(u, 0.25))
        
        # Aggiornamento dello stato dell'osservatore per il passo successivo
        u_excess = max(0.0, u - self.basal_rate)
        ds1 = (u_excess - (self.s1 / self.tau_s)) * self.dt
        ds2 = ((self.s1 / self.tau_s) - (self.s2 / self.tau_s)) * self.dt
        self.s1 += ds1
        self.s2 += ds2
        self.prev_cgm = cgm
        
        return Action(basal=u, bolus=0.0)

    def reset(self):
        self.integral_error = 0.0
        self.prev_cgm = None
        self.s1 = 0.0
        self.s2 = 0.0
        self.current_iob = 0.0

## 2. Configurazione ed Esecuzione della Simulazione Comparativa (16 Ore)

In [ ]:
def run_simulation(controller_class, patient_name='adult#001', meal_cho=60.0):
    # Definizione temporale: 16 ore (dalle 08:00 alle 00:00)
    start_time = datetime(2026, 1, 1, 8, 0, 0)
    sim_hours = 16
    meal_scenario = [(12.0, meal_cho)]  # Pasto alle ore 12:00 (dopo 4h dall'avvio)
    
    patient = T1DPatient.withName(patient_name)
    sensor = CGMSensor.withName('Dexcom', seed=1)
    pump = InsulinPump.withName('Insulet')
    scenario = CustomScenario(start_time=start_time, scenario=meal_scenario)
    env = T1DSimEnv(patient, sensor, pump, scenario)
    
    dt = env.sample_time  # 3 minuti
    total_steps = int(sim_hours * 60 / dt)  # 320 passi esatti
    
    # Estrazione basale nominale per il paziente
    bw = patient._params['BW']
    u2ss = patient._params['u2ss']
    basal_nominal = (u2ss * bw) / 6000.0
    
    controller = controller_class(basal_rate=basal_nominal, dt=dt)
    
    # Strutture dati di log
    time_list = []
    cgm_list = []
    bg_list = []
    insulin_list = []
    iob_list = []
    
    obs, reward, done, info = env.reset()
    controller.reset()
    
    for step in range(total_steps):
        current_t = start_time + timedelta(minutes=step * dt)
        action = controller.policy(obs, reward, done, **info)
        
        time_list.append(current_t)
        cgm_list.append(obs.CGM)
        bg_list.append(env.patient.state[0])  # Glucosio plasmatico
        insulin_list.append(action.basal * 60.0)  # Convertito in U/h per visualizzazione
        
        if hasattr(controller, 'current_iob'):
            iob_list.append(controller.current_iob)
        else:
            iob_list.append(0.0)
            
        obs, reward, done, info = env.step(action)
        if done:
            break
            
    df_res = pd.DataFrame({
        'Time': time_list,
        'CGM': cgm_list,
        'BG': bg_list,
        'Insulin_Uh': insulin_list,
        'IOB': iob_list
    })
    return df_res

# Esecuzione dei due scenari
df_pid = run_simulation(StandardPIDController, patient_name='adult#001', meal_cho=60.0)
df_iob = run_simulation(IOBConstrainedController, patient_name='adult#001', meal_cho=60.0)
print("Simulazioni nominali a 16 ore completate con successo.")

## 3. Calcolo degli Indicatori Clinici di Consenso e Ipoglicemia Tardiva

Le metriche includono:
- **TIR** ($70 - 180\text{ mg/dL}$)
- **TAR Livello 1** ($181 - 250\text{ mg/dL}$) e **TAR Livello 2** ($> 250\text{ mg/dL}$)
- **TBR Livello 1** ($54 - 69\text{ mg/dL}$) e **TBR Livello 2** ($< 54\text{ mg/dL}$)
- **TBR Tardivo ($3 - 6\text{ ore}$ postprandiale)**: percentuale di ipoglicemia nella finestra $15:00 - 18:00$, specifica per quantificare l'*insulin stacking*.

In [ ]:
def compute_clinical_metrics(df, label):
    cgm = df['CGM'].values
    n = len(cgm)
    
    tir = np.sum((cgm >= 70.0) & (cgm <= 180.0)) / n * 100.0
    tar_l1 = np.sum((cgm > 180.0) & (cgm <= 250.0)) / n * 100.0
    tar_l2 = np.sum(cgm > 250.0) / n * 100.0
    tar_tot = tar_l1 + tar_l2
    
    tbr_l1 = np.sum((cgm >= 54.0) & (cgm < 70.0)) / n * 100.0
    tbr_l2 = np.sum(cgm < 54.0) / n * 100.0
    tbr_tot = tbr_l1 + tbr_l2
    
    peak = np.max(cgm)
    nadir = np.min(cgm)
    
    # Finestra tardiva: 3-6 ore dopo il pasto (dalle ore 15:00 alle 18:00)
    mask_late = (df['Time'] >= datetime(2026, 1, 1, 15, 0, 0)) & \
                (df['Time'] <= datetime(2026, 1, 1, 18, 0, 0))
    cgm_late = df.loc[mask_late, 'CGM'].values
    tbr_late = np.sum(cgm_late < 70.0) / len(cgm_late) * 100.0 if len(cgm_late) > 0 else 0.0
    
    return {
        'Controllore': label,
        'TIR [%]': round(tir, 2),
        'TAR Tot [%]': round(tar_tot, 2),
        'TAR L1 [%]': round(tar_l1, 2),
        'TAR L2 [%]': round(tar_l2, 2),
        'TBR Tot [%]': round(tbr_tot, 2),
        'TBR L1 [%]': round(tbr_l1, 2),
        'TBR L2 [%]': round(tbr_l2, 2),
        'TBR Tardivo (3-6h) [%]': round(tbr_late, 2),
        'Picco [mg/dL]': round(peak, 1),
        'Nadir [mg/dL]': round(nadir, 1)
    }

m_pid = compute_clinical_metrics(df_pid, 'PID Baseline')
m_iob = compute_clinical_metrics(df_iob, 'PID + Vincolo IOB + PLGS')

df_metrics = pd.DataFrame([m_pid, m_iob])
display(df_metrics)

## 4. Grafico a 3 Pannelli Sincronizzato

Rappresentazione grafica richiesta dalle specifiche cliniche:
1. **Pannello Superiore**: Glicemia interstiziale CGM con evidenziazione della fascia euglicemica ($70-180\text{ mg/dL}$).
2. **Pannello Mediano**: Portata di infusione ormonale erogata dal microinfusore ($u(t)$ in $\text{U/h}$).
3. **Pannello Inferiore**: Profilo di stima dell'Insulina Attiva residua ($IOB(t)$ in $\text{U}$).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Formattazione asse temporale in ore dall'avvio
t_hours = [(t - df_pid['Time'].iloc[0]).total_seconds() / 3600.0 for t in df_pid['Time']]

# --- 1. GLICEMIA (CGM) ---
ax1 = axes[0]
ax1.axhspan(70, 180, color='green', alpha=0.15, label='Fascia Euglicemica (70-180 mg/dL)')
ax1.axhline(70, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.axhline(180, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax1.axvline(4.0, color='black', linestyle=':', linewidth=1.5, label='Pasto non annunciato (60g @ 12:00)')
ax1.plot(t_hours, df_pid['CGM'], 'r--', linewidth=2, label='PID Baseline')
ax1.plot(t_hours, df_iob['CGM'], 'b-', linewidth=2, label='PID con Vincolo IOB + PLGS')
ax1.set_ylabel('CGM [mg/dL]', fontsize=11)
ax1.set_title('Risposta Dinamica a Disturbo Prandiale non Annunciato (Paziente adult#001)', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right')

# --- 2. INFUSIONE INSULINICA ---
ax2 = axes[1]
ax2.axvline(4.0, color='black', linestyle=':', linewidth=1.5)
ax2.plot(t_hours, df_pid['Insulin_Uh'], 'r--', linewidth=1.8, label='Infusione PID Baseline')
ax2.plot(t_hours, df_iob['Insulin_Uh'], 'b-', linewidth=1.8, label='Infusione con Vincolo IOB + PLGS')
ax2.set_ylabel('Erogazione [U/h]', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper right')

# --- 3. INSULINA ATTIVA (IOB) ---
ax3 = axes[2]
ax3.axvline(4.0, color='black', linestyle=':', linewidth=1.5)
ax3.axhline(2.2, color='darkblue', linestyle='-.', alpha=0.8, label='Soglia di saturazione IOB (Cap = 2.2 U)')
ax3.plot(t_hours, df_iob['IOB'], 'b-', linewidth=2, label='Stima IOB (Controllore Vincolato)')
ax3.set_ylabel('IOB Stimato [U]', fontsize=11)
ax3.set_xlabel('Tempo dalla partenza della simulazione [ore]', fontsize=11)
ax3.set_xlim([0, 16])
ax3.grid(True, linestyle=':', alpha=0.6)
ax3.legend(loc='upper right')

plt.tight_layout()
plt.savefig('grafico_benchmark_baseline.png', dpi=300)
plt.show()
print("Grafico a 3 pannelli generato ed esportato in grafico_benchmark_baseline.png.")